In [10]:
import numpy as np
import pandas as pd

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [12]:
df = pd.read_csv("train.csv", usecols=['Age', 'Fare', 'SibSp', 'Parch', 'Survived'])

In [13]:
df.shape

(891, 5)

In [14]:
df.dropna(inplace=True)

In [15]:
df.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [16]:
df.shape

(714, 5)

In [17]:
# i will combine siblingspouse(sibsp) and parentchild(parch)
df['family'] = df['SibSp'] + df['Parch']

In [18]:
df.head()

,Survived,Age,SibSp,Parch,Fare,family
0,0,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,1
2,1,26.0,0,0,7.9250,0
3,1,35.0,1,0,53.1000,1
4,0,35.0,0,0,8.0500,0


In [20]:
# now i dont need sibsp and parch so i will drop these since i have family
df.drop(columns=['SibSp', 'Parch'], inplace=True)

In [21]:
df.head()

,Survived,Age,Fare,family
0,0,22.0,7.2500,1
1,1,38.0,71.2833,1
2,1,26.0,7.9250,0
3,1,35.0,53.1000,1
4,0,35.0,8.0500,0


In [22]:
# now lets take x and y
X = df.drop(columns=['Survived'])
y = df['Survived']

In [23]:
# lets do preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
X_train.head(10)

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0
30,40.0,27.7208,0
287,22.0,7.8958,0
217,42.0,27.0000,1
797,31.0,8.6833,0
371,18.0,6.4958,1


In [25]:
y_train

,Survived
328,1
73,0
253,0
719,0
666,0
...,...
92,0
134,0
337,1
548,0


In [27]:
# Without using binarization
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy_score(y_test, y_pred)

0.6433566433566433

In [36]:
# lets confirm using cross_val_score
from sklearn.model_selection import cross_val_score
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy'))

np.float64(0.648611111111111)

# Applying Binarization

In [37]:
from sklearn.preprocessing import Binarizer

In [38]:
# forming the transformer
trf = ColumnTransformer([
    ('bin', Binarizer(copy=False),['family'])
], remainder = 'passthrough')

In [39]:
# now raing with the transformed data
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [41]:
# now converting them into dataframes
pd.DataFrame(X_train_trf, columns=['family', 'Age', 'Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [51]:
# now again using the decision tree
clf = DecisionTreeClassifier()
clf.fit(X_train_trf, y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test, y_pred2)

0.6013986013986014

In [52]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy'))

np.float64(0.6499608763693271)